In [2]:
from z3 import *


In [12]:
# 1. Which input triggers the branch?
# if x + 10 == 3: BRANCH using BitVec

x = BitVec('x', 8)

t = Then('simplify', 'solve-eqs').solver()

solve_using(t, x + 10 == 3)


[x = 249]


In [7]:
# 2. Propagate values

x, y, z = Ints('x y z')

g = Goal()
g.add(x == 5)
g.add(y == x + 2)
g.add(z > y)
g.add(z < 20)

t = Tactic('propagate-values')

result = t(g)
print(g)
print(result[0])
print(len(g), len(result[0]))

t2 = Tactic('solve-eqs')
result = t2(g)
print(g)
print(result[0])
print(len(g), len(result[0]))

solver = Then('simplify', 'solve-eqs', 'smt').solver()
solver.add(x == 5, y == x + 2, z > y, z < 20)
print(solver.check())
print(solver.model())


[x == 5, y == x + 2, z > y, z < 20]
[x == 5, y == 7, Not(z <= 7), Not(20 <= z)]
4 4
[x == 5, y == x + 2, z > y, z < 20]
[Not(z <= 7), Not(20 <= z)]
4 2
[x == 5, y == x + 2, z > y, z < 20]
[Not(z <= 7), Not(20 <= z)]
4 2
sat
[z = 8, y = 7, x = 5]


In [14]:
# 3. OrElse
# x > 3, y > x

x, y = Ints('x y')

g = Goal()
g.add(x > 3)
g.add(y > x)

t1 = Tactic('solve-eqs')

result = t1(g)
print(result)
print(result[0])

t2 = Tactic('split-clause')
# print(t2(g)) raises because split structure is not present

t3 = OrElse('split-clause', 'skip')
print(t3(g))


[[x > 3, y > x]]
[x > 3, y > x]
[[x > 3, y > x]]


In [18]:
# 4. Num Consts

x, y, z, w = Ints('x y z w')

small_goal = Goal()
small_goal.add(x > 1, y == x + 1)

big_goal = Goal()
big_goal.add(x > 1, y == x + 1, z == y + 1, w > z)

variable_count = Probe('num-consts')
t = If(variable_count > 3, Tactic('solve-eqs'), Tactic('skip'))

print(variable_count(small_goal))
print(variable_count(big_goal))
print(t(small_goal))
print(t(big_goal))


2.0
4.0
[[x > 1, y == x + 1]]
[[Not(z <= 3), Not(w <= z)]]


In [23]:
# 5. Mini strategy

g = Goal()
g.add(Or(x == 2, x == 5))
g.add(y == x + 1)
g.add(z > y)
g.add(z < 10)

t = Then(
    Repeat(OrElse('split-clause', 'skip')),
    'propagate-values',
    'solve-eqs'
)

print(t(g))

solver = t.solver()
solver.add(g)

print(solver.check())
print(solver.model())

t = Then(
    Repeat(OrElse('split-clause', 'skip')),
    'propagate-values',
    'solve-eqs',
    'smt'
)

solver = t.solver()
solver.add(g)
print(solver.check())
print(solver.model())


[[Not(z <= 3), Not(10 <= z)], [Not(z <= 6), Not(10 <= z)]]
unknown
[x = 2, y = 3]
sat
[z = 4, x = 2, y = 3]
